In [15]:
import os
import json
import csv

def extract_data_from_annotation(annotation_file):
    # Function to extract relative_stance from the annotation file
    with open(annotation_file, 'r') as file:
        annotation_data = json.load(file)
        print(f"Annotation Data {annotation_data}")
        article_lvl_ann = annotation_data.get('article-level-annotations', "")
        stance_ann = article_lvl_ann.get("relative_stance","")
        print(f"STANCE: {stance_ann}")
        return stance_ann

def extract_data_from_article(article_file):
    # Function to extract title and body from the article file
    with open(article_file, 'r') as file:
        article_data = json.load(file)
        title = article_data.get("title", "")
        body_paragraphs = [" ".join(paragraph) for paragraph in article_data.get("body-paragraphs", [])]
        return title, "\n".join(body_paragraphs)
    
def generate_split_csv(datafolder):    
    social_value_filename = 'processed_data_social.csv'
    economic_value_filename = 'processed_data_economic.csv'
    social_values = ['liberal', 'conservative']
    econ_values = ['left', 'right', ]

    triplets = {}

    # Annotations are separated into years in the annotations folder
    for year in os.listdir(os.path.join(datafolder, 'annotations')):
        for annotation_file in os.listdir(os.path.join(datafolder, 'annotations', year)):
            if annotation_file.endswith('_ann.json'):
                triplet_uuid = annotation_file.split('_',2)[0]
                print(triplet_uuid)
                annotation_path = os.path.join(datafolder, 'annotations', year, annotation_file)

                stance = extract_data_from_annotation(annotation_path)
                print(f"Annotation file name : {annotation_path} - {stance}")
                
                if triplet_uuid not in triplets:
                    triplets[triplet_uuid] = []
                triplets[triplet_uuid].append(stance)


    # Function to generate the CSV file with the desired columns
    with open(social_value_filename, 'w', newline='', encoding='utf-8') as socialfile:
        with open (economic_value_filename, 'w', newline='', encoding='utf-8') as econfile:
            social_writer = csv.writer(socialfile)
            social_writer.writerow(['title', 'body', 'stance'])  # Writing header

            econ_writer = csv.writer(econfile)
            econ_writer.writerow(['title', 'body', 'stance'])  # Writing header

            # Annotations are separated into years in the annotations folder
            for year in os.listdir(os.path.join(datafolder, 'articles')):
                for article_file in os.listdir(os.path.join(datafolder, 'articles', year)):
                    if article_file.endswith('.json'):
                        triplet_uuid = article_file.split('_',2)[0]
                        print(triplet_uuid)

                        title, body = extract_data_from_article(os.path.join(datafolder, 'articles', year, article_file))

                        if triplet_uuid in triplets:
                            stances = triplets[triplet_uuid]
                            is_social = any([stance in social_values for stance in stances])
                            is_econ = any([stance in econ_values for stance in stances])
                            if is_social and is_econ:
                                print(f'ERROR: {triplet_uuid} is both social and econ')
                            elif is_social:
                                social_writer.writerow([title, body, stance])
                            elif is_econ:
                                econ_writer.writerow([title, body, stance])
                            else:
                                print(f'{triplet_uuid} has no stance')
                                social_writer.writerow([title, body, stance])
                                econ_writer.writerow([title, body, stance])

def generate_csv(datafolder, csv_file):    

    # Function to generate the CSV file with the desired columns
    with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(['title', 'body', 'stance'])  # Writing header

        # Annotations are separated into years in the annotations folder
        for year in os.listdir(os.path.join(datafolder, 'annotations')):
            for annotation_file in os.listdir(os.path.join(datafolder, 'annotations', year)):
                if annotation_file.endswith('_ann.json'):
                    triplet_uuid = annotation_file.split('_',2)[:2]
                    print(triplet_uuid)
                    article_file = triplet_uuid[0] + '_'+ triplet_uuid[1] + '.json'
                    print(article_file)
                    annotation_path = os.path.join(datafolder, 'annotations', annotation_file)
                    print(f"Annotation file name : {annotation_path}")

                    article_path = os.path.join(datafolder, 'articles', year, article_file)
                    if os.path.exists(article_path):
                        print('PATH EXISTS')
                        # If the corresponding article file exists, extract data
                        stance = extract_data_from_annotation(annotation_path)
                        title, body = extract_data_from_article(article_path)
                        if stance == 'liberal':
                            stance = 'left'
                        elif stance == 'conservative':
                            stance = 'right'

                        # Write data to the CSV file
                        csv_writer.writerow([title, body, stance])




In [16]:
if __name__ == "__main__":
    datafolder = "BASIL"
    csv_file = "processed_data_combined.csv"
    generate_csv(datafolder, csv_file)

FileNotFoundError: [Errno 2] No such file or directory: '2013'